# Hindi ASR — Whisper + PEFT LoRA Fine-tuning on FLEURS

This notebook trains `openai/whisper-small` with PEFT LoRA adapters on the FLEURS Hindi dataset.
- **Dataset**: `google/fleurs` `hi_in` — ~1,296 real multi-speaker Hindi speech samples
- **Method**: PEFT LoRA (r=16) on decoder attention layers — 1.77M trainable params (0.73%)
- **Output**: ~3.4 MB LoRA adapter checkpoint pushed to HuggingFace Hub
- **Runtime**: ~2–3 hours on Colab T4 GPU

**Before running**: Set Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"CUDA: {torch.version.cuda}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────
!git clone https://github.com/27Kushal/Hindi-asr-whisper.git
%cd Hindi-asr-whisper

In [ ]:
# ── 3. Install dependencies ───────────────────────────────
!pip install -r requirements.txt -q
# Verify key packages
import peft, transformers, datasets, evaluate
print(f"peft: {peft.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets: {datasets.__version__}")

In [ ]:
# ── 4. (Optional) HuggingFace login to push model later ──
# Get your token from https://huggingface.co/settings/tokens
from huggingface_hub import login
login()  # Enter your HF token when prompted

In [ ]:
# ── 5. Verify FLEURS loads correctly ─────────────────────
from datasets import load_dataset
sample_ds = load_dataset('google/fleurs', 'hi_in', split='train', streaming=True, trust_remote_code=True)
sample = next(iter(sample_ds))
print('FLEURS sample fields:', list(sample.keys()))
print('Transcription:', sample.get('transcription', '')[:100])
print('Audio sample rate:', sample['audio']['sampling_rate'])
print('FLEURS OK ✓')

In [ ]:
# ── 6. Run Mode A: Base Whisper zero-shot baseline ───────
# Quick evaluation (~5 min) — establishes the starting point
!python scripts/run_ablation.py --config configs/config.yaml --modes A
!cat models/ablation/base_whisper/test_results.json

In [ ]:
# ── 7. Run Mode C: LoRA PEFT training (main run) ─────────
# ~2–3 hours on T4 GPU with FP16
!CUDA_VISIBLE_DEVICES=0 python train.py \
    --config configs/config.yaml \
    --dataset google/fleurs \
    2>&1 | tee training_log.txt

In [ ]:
# ── 8. Run Mode B: Encoder-frozen ablation ───────────────
# ~2–3 hours on T4 GPU — comparison baseline
!CUDA_VISIBLE_DEVICES=0 python train.py \
    --config configs/config.yaml \
    --dataset google/fleurs \
    --no_lora \
    2>&1 | tee training_log_frozen.txt

In [ ]:
# ── 9. Generate ablation comparison table ────────────────
!python scripts/run_ablation.py --config configs/config.yaml --modes A B C
!python scripts/generate_report.py

In [ ]:
# ── 10. Cross-dataset benchmark (Common Voice Hindi) ─────
# Tests out-of-distribution generalization
!python scripts/eval_benchmark.py \
    --checkpoint ./models/whisper-lora-hindi/final \
    --dataset all \
    --language hindi \
    --language_code hi_in

In [ ]:
# ── 11. INT8 Quantization (CUDA path) ────────────────────
!python scripts/export_quantized.py \
    --checkpoint ./models/whisper-lora-hindi/final \
    --device cuda
!cat models/whisper-lora-hindi/quantized/quantization_report.json

In [ ]:
# ── 12. Push LoRA adapter to HuggingFace Hub ─────────────
# The adapter is only ~3.4 MB — fast upload
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

HF_REPO = 'kushalbagla/whisper-small-hindi-lora'  # Change to your username

base = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')
model = PeftModel.from_pretrained(base, './models/whisper-lora-hindi/final')
processor = WhisperProcessor.from_pretrained('./models/whisper-lora-hindi/final')

model.push_to_hub(HF_REPO)
processor.push_to_hub(HF_REPO)
print(f'Model pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 13. Print final results summary ──────────────────────
import json

print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)

for label, path in [
    ('LoRA PEFT (FLEURS test)', 'models/whisper-lora-hindi/test_results.json'),
    ('Common Voice (OOD)', 'models/whisper-lora-hindi/final/eval_common_voice.json'),
    ('FLEURS in-dist', 'models/whisper-lora-hindi/final/eval_fleurs.json'),
]:
    try:
        with open(path) as f:
            r = json.load(f)
        print(f'\n{label}:')
        print(f'  WER: {r.get("wer", r.get("test_wer", "?")):.4f}')
        print(f'  CER: {r.get("cer", r.get("test_cer", "?")):.4f}')
    except FileNotFoundError:
        print(f'\n{label}: not found')

print('\nCheckpoint size:')
import os
total = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk('models/whisper-lora-hindi/final')
    for f in files if f.endswith(('.bin', '.safetensors'))
) / (1024**2)
print(f'  {total:.1f} MB')
print('='*60)